# Video Embedding Setup

이 노트북에서는 이후 노트북(02, 03, 04)에서 사용할 비디오 임베딩을 준비합니다.

Workshop 1(`1_basic`)에서 이미 임베딩을 생성했다면 **자동으로 감지하여 재사용**합니다. 기존 임베딩이 없는 경우에만 Marengo 3.0을 통해 새로 생성합니다.

직접 준비한 비디오로 테스트하려면 `videos/` 폴더에 `.mp4` 파일을 넣어주세요.

## 0. Setup

In [ ]:
%pip install -r requirements.txt -Uq

In [ ]:
import json, time, os
import boto3
from botocore.config import Config

session = boto3.Session()
REGION = session.region_name or "us-east-1"
AWS_ACCOUNT_ID = session.client("sts").get_caller_identity()["Account"]
bedrock = session.client("bedrock-runtime", region_name=REGION, config=Config(signature_version="v4"))
s3 = session.client("s3", region_name=REGION)

print(f"✅ Region: {REGION}, Account: {AWS_ACCOUNT_ID}")

## 1. Workshop Resources

CloudFormation 출력에서 제공된 워크샵 리소스를 입력합니다. 이 값들은 `config.json`에 저장되어 이후 노트북에서 자동으로 로드됩니다.

In [ ]:
# ==============================
# TODO: 워크샵 리소스를 입력하세요
# ==============================
S3_BUCKET = "<YOUR_S3_BUCKET_NAME>"                          # WorkshopS3BucketName
OPENSEARCH_ENDPOINT = "<YOUR_OPENSEARCH_ENDPOINT>"           # OpenSearchVectorDBEndpoint
S3_VECTOR_BUCKET_NAME = "<YOUR_S3_VECTOR_BUCKET_NAME>"       # WorkshopS3VectorBucketName

# Validate
if S3_BUCKET.startswith("<"):
    raise ValueError("S3_BUCKET을 입력해주세요 (CloudFormation 출력의 WorkshopS3BucketName)")
if OPENSEARCH_ENDPOINT.startswith("https://"):
    OPENSEARCH_ENDPOINT = OPENSEARCH_ENDPOINT.replace("https://", "")
if OPENSEARCH_ENDPOINT.startswith("<"): OPENSEARCH_ENDPOINT = ""
if S3_VECTOR_BUCKET_NAME.startswith("<"): S3_VECTOR_BUCKET_NAME = ""

VIDEO_S3_PREFIX = "vectordb-workshop/videos/"
EMBEDDINGS_S3_PREFIX = "vectordb-workshop/embeddings/"
MODEL_ID = "twelvelabs.marengo-embed-3-0-v1:0"

print(f"✅ S3 Bucket: {S3_BUCKET}")
print(f"   OpenSearch: {OPENSEARCH_ENDPOINT or '(미입력)'}")
print(f"   S3 Vector Bucket: {S3_VECTOR_BUCKET_NAME or '(미입력)'}")

## 2. Detect Existing Embeddings

S3에서 기존 임베딩 파일을 탐색합니다. Workshop 1에서 생성한 임베딩이 있으면 자동으로 재사용합니다.

In [ ]:
def find_existing_embeddings():
    """Search for existing embedding files across known S3 paths."""
    for prefix in [EMBEDDINGS_S3_PREFIX, "embeddings/videos/", "embeddings/"]:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=100)
        files = [o for o in resp.get("Contents", []) if o["Key"].endswith(".json")]
        if files:
            return prefix, files
    return None, []

def load_embedding_file(s3_key):
    """Load embedding file (handles Workshop 1 and 4 formats)."""
    data = json.loads(s3.get_object(Bucket=S3_BUCKET, Key=s3_key)["Body"].read().decode())
    if "vectors" in data: return data["vectors"]
    if "data" in data: return data["data"]
    vecs = []
    for rec in data.get("embeddings", [data]):
        for seg in rec.get("data", []):
            if "embedding" in seg: vecs.append(seg)
    return vecs

found_prefix, found_files = find_existing_embeddings()
EMBEDDINGS_EXIST = False
all_embeddings = {}

if found_files:
    print(f"✅ 기존 임베딩 {len(found_files)}개 파일 발견: s3://{S3_BUCKET}/{found_prefix}")
    for f in found_files:
        vecs = load_embedding_file(f["Key"])
        if not vecs or "embedding" not in vecs[0]: continue
        basename = os.path.splitext(os.path.basename(f["Key"]))[0]
        vid = f["Key"].split("/")[-3] if basename == "output" else basename
        all_embeddings[vid] = {"video_id": vid, "vectors": vecs}
        print(f"   {vid}: {len(vecs)} vectors")
    if all_embeddings:
        EMBEDDINGS_EXIST = True
        EMBEDDINGS_S3_PREFIX = found_prefix
        print(f"\n✅ 기존 임베딩 재사용 → Section 5로 이동하세요")

if not EMBEDDINGS_EXIST:
    print("ℹ️  기존 임베딩 없음 — Section 3에서 비디오 업로드 후 Section 4에서 생성합니다")

## 3. Upload Videos to S3

**기존 임베딩이 있으면 이 섹션은 건너뛰세요.**

`videos/` 폴더에 `.mp4` 파일을 넣으면 자동으로 S3에 업로드됩니다.

In [ ]:
if EMBEDDINGS_EXIST:
    print("⏭️  기존 임베딩 사용 — 비디오 업로드 건너뜀")
else:
    LOCAL_VIDEO_DIR = "videos"
    resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=VIDEO_S3_PREFIX, MaxKeys=100)
    existing = {os.path.basename(o["Key"]) for o in resp.get("Contents", []) if o["Key"].endswith(".mp4")}

    if os.path.isdir(LOCAL_VIDEO_DIR):
        for vf in sorted(os.listdir(LOCAL_VIDEO_DIR)):
            if not vf.endswith(".mp4"): continue
            if vf in existing:
                print(f"   ⏭️  {vf} (이미 S3에 존재)")
            else:
                print(f"   ⬆️  {vf} 업로드 중...", end=" ")
                s3.upload_file(os.path.join(LOCAL_VIDEO_DIR, vf), S3_BUCKET, f"{VIDEO_S3_PREFIX}{vf}")
                print("✅")

    resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=VIDEO_S3_PREFIX, MaxKeys=100)
    video_keys = [o["Key"] for o in resp.get("Contents", []) if o["Key"].endswith(".mp4")]
    print(f"\n{'✅' if video_keys else '❌'} S3 비디오: {len(video_keys)}개")

## 4. Generate Embeddings

**기존 임베딩이 있으면 이 섹션은 건너뛰세요.**

Marengo 3.0은 비디오를 시간 단위 세그먼트로 나누어 각각 512차원 벡터를 생성합니다.
- **clip-level**: 각 세그먼트별 임베딩
- **asset-level**: 전체 비디오 요약 임베딩

In [ ]:
EMBEDDING_MODALITY = "visual"
POLL_INTERVAL = 15

In [ ]:
if EMBEDDINGS_EXIST:
    print("⏭️  기존 임베딩 사용 — 생성 건너뜀")
else:
    def start_embedding_job(video_s3_key):
        video_id = os.path.splitext(os.path.basename(video_s3_key))[0]
        resp = bedrock.start_async_invoke(
            modelId=MODEL_ID,
            modelInput={"inputType": "video", "video": {
                "mediaSource": {"s3Location": {"uri": f"s3://{S3_BUCKET}/{video_s3_key}", "bucketOwner": AWS_ACCOUNT_ID}},
                "embeddingOption": [EMBEDDING_MODALITY], "embeddingScope": ["clip", "asset"]
            }},
            outputDataConfig={"s3OutputDataConfig": {"s3Uri": f"s3://{S3_BUCKET}/{EMBEDDINGS_S3_PREFIX}{video_id}/"}}
        )
        return resp["invocationArn"], video_id

    def wait_for_job(arn, timeout=600):
        start = time.time()
        while time.time() - start < timeout:
            job = bedrock.get_async_invoke(invocationArn=arn)
            status = job["status"]
            if status == "Completed": return job
            if status in ("Failed", "Expired"): return None
            print(f"  [{time.strftime('%H:%M:%S')}] {status}"); time.sleep(POLL_INTERVAL)
        return None

    def download_embeddings(video_id):
        prefix = f"{EMBEDDINGS_S3_PREFIX}{video_id}/"
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix)
        results = []
        for obj in resp.get("Contents", []):
            if obj["Key"].endswith((".json", ".jsonl")):
                raw = s3.get_object(Bucket=S3_BUCKET, Key=obj["Key"])["Body"].read().decode()
                for line in (raw.strip().split("\n") if obj["Key"].endswith(".jsonl") else [raw]):
                    if line.strip(): results.append(json.loads(line))
        return results

    resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=VIDEO_S3_PREFIX)
    video_keys = [o["Key"] for o in resp.get("Contents", []) if o["Key"].endswith(".mp4")]

    jobs = []
    for vk in video_keys:
        print(f"  시작: {os.path.basename(vk)}...", end=" ")
        try:
            arn, vid = start_embedding_job(vk); jobs.append({"arn": arn, "video_id": vid}); print("✅")
        except Exception as e: print(f"❌ {e}")

    for job in jobs:
        vid = job["video_id"]
        print(f"\n대기 중: {vid}...")
        if wait_for_job(job["arn"]):
            raw = download_embeddings(vid)
            vectors = [seg for rec in raw for seg in rec.get("data", []) if "embedding" in seg]
            all_embeddings[vid] = {"video_id": vid, "vectors": vectors}
            print(f"  ✅ {len(vectors)} vectors")

    for vid, data in all_embeddings.items():
        s3.put_object(Bucket=S3_BUCKET, Key=f"{EMBEDDINGS_S3_PREFIX}{vid}.json",
                      Body=json.dumps(data), ContentType="application/json")
    print(f"\n✅ 총 {sum(len(v['vectors']) for v in all_embeddings.values())}개 벡터 생성 완료")

## 5. Save Configuration

임베딩 구조를 확인하고 `config.json`을 저장합니다. 이후 노트북(02, 03, 04)에서 이 파일을 자동으로 로드합니다.

In [ ]:
for vid, data in all_embeddings.items():
    vectors = data["vectors"]
    asset = [v for v in vectors if v.get("embeddingScope") == "asset"]
    clips = [v for v in vectors if v.get("embeddingScope") == "clip"]
    print(f"📹 {vid}: {len(vectors)} vectors (asset={len(asset)}, clip={len(clips)})")

first_vid = next(iter(all_embeddings.values()))
EMBEDDING_DIMENSION = len(first_vid["vectors"][0]["embedding"])

config = {
    "REGION": REGION, "S3_BUCKET": S3_BUCKET, "AWS_ACCOUNT_ID": AWS_ACCOUNT_ID,
    "EMBEDDINGS_S3_PREFIX": EMBEDDINGS_S3_PREFIX, "EMBEDDING_DIMENSION": EMBEDDING_DIMENSION,
    "VIDEO_IDS": list(all_embeddings.keys()),
    "OPENSEARCH_ENDPOINT": OPENSEARCH_ENDPOINT, "S3_VECTOR_BUCKET_NAME": S3_VECTOR_BUCKET_NAME,
    "VIDEO_S3_PREFIX": VIDEO_S3_PREFIX
}
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"\n✅ config.json 저장 완료 (dimension={EMBEDDING_DIMENSION}, videos={len(all_embeddings)})")
print(f"   다음 단계: 02_opensearch_serverless.ipynb 또는 03_s3_vectors.ipynb 실행")